# Adapting a trained SpliceVI model to a new, sparse atlas

This tutorial takes a SpliceVI model trained on a deep, full-length reference atlas (Allen Brain, Smart-seq) and adapts it
to a sparse query atlas (EasySci, single-nucleus 3′ sequencing) with `SPLICEVI.finetune_decoders`. This is how the
paper's EasySci model was made.

**The idea.** The reference model has learned two things:

- **encoders**, which place a cell in a latent space from its expression and splicing;
- **decoders**, which turn a latent position back into expected expression and junction usage (PSI).

`finetune_decoders` keeps the encoders, and so the latent space, exactly as they are, and retrains only the decoders on
the query cells. It also adds one decoder input, set to 0 for reference cells and 1 for query cells, which lets the
decoders learn how the query technology differs. Reference cells are mixed into every step (a *replay set*) so the
decoders keep fitting the reference too.

**When to use it.** Your atlas has too few reads per junction to train a splicing model on its own, but a related deep
atlas with a trained SpliceVI model exists. Query cells then borrow the reference model's view of cell identity, while
the decoders learn the query data's own splicing.

**What you need**
- a trained SpliceVI model (here: Allen Brain, deviation input, replicate 1 of the paper);
- your query data as a SpliceVI MuData in the **same feature space** as that model (same genes and junctions, same order);
- a replay set: a few thousand of the reference model's training cells.

**Steps**
1. Load the data and check the feature space.
2. Fine-tune the decoders.
3. Save and reload the model.
4. Look at the query cells in the latent space.
5. Find differential splicing between two cell types, and check one example against the raw reads.
6. Compare conditions within one cell type.
7. Decode query cells as if they were reference data.

A GPU is strongly recommended for step 2 (about 30 s per epoch on one GPU for this subset).

> **Data.** The files below are on the Knowles lab cluster for now. They will shortly be replaced by a Zenodo
> download, and only `DATA_DIR` and `BASE_MODEL` will change.
>
> - `easysci_in80_tutorial.h5mu`: 6,977 EasySci metacells (pseudo-bulks of similar nuclei, median 34 nuclei each) of four
>   cell types: cortical projection neurons, interneurons, oligodendrocytes and Purkinje neurons. These are the paper's
>   EasySci training metacells (*in80*: built from 80% of each metacell's nuclei; the other 20% were kept unseen for
>   validation).
> - `allen_replay_5k.h5mu`: 5,000 Allen Brain training cells, the replay set (the paper used 20,000).
> - `pipeline_20261001_012113_s5-ab-devp_rep1/`: the Allen Brain SpliceVI model.

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import torch

from splicevi import SPLICEVI, filter_read_support, preset

# Paths on the Knowles lab cluster (to be replaced by a Zenodo download)
DATA_DIR = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI_Repositories/script_outputs/tutorial_data"
BASE_MODEL = "/gpfs/commons/home/svaidyanathan/repos/SpliceVI_Repositories/SpliceVI/models/pipeline_20261001_012113_s5-ab-devp_rep1"
OUT_DIR = "finetuned_easysci_tutorial"   # where the fine-tuned model is saved

QUERY_FILE = os.path.join(DATA_DIR, "easysci_in80_tutorial.h5mu")
REFERENCE_FILE = os.path.join(DATA_DIR, "allen_replay_5k.h5mu")

MAX_EPOCHS = 30   # a short run for the tutorial; the paper used the defaults (up to 300 epochs, early stopping after 30 without improvement)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

The next cell defines a few helpers: loading a MuData, the feature-space check, raw PSI from read counts, and two plots.
It is collapsed; expand it to see the code.

In [ ]:
# Helpers: data loading, feature-space check, raw PSI from read counts, plots
import matplotlib.pyplot as plt
import mudata as mu


def load_mudata(path):
    # Read a SpliceVI MuData; copy obs columns and the library size into the rna modality, where scvi-tools looks for them.
    mdata = mu.read_h5mu(path)
    for column in mdata.obs.columns:
        if column not in mdata["rna"].obs.columns:
            mdata["rna"].obs[column] = mdata.obs[column].values
    if "X_library_size" in mdata["rna"].obsm:
        mdata["rna"].obs["X_library_size"] = np.asarray(mdata["rna"].obsm["X_library_size"]).ravel()
    return mdata


def check_feature_space(query, reference):
    # The query must have the reference model's genes and junctions, in the same order.
    same_genes = query["rna"].var_names.equals(reference["rna"].var_names)
    same_junctions = np.array_equal(query["splicing"].var["junction_id"].astype(str).values,
                                    reference["splicing"].var["junction_id"].astype(str).values)
    print(f"genes: {query['rna'].n_vars:,} query vs {reference['rna'].n_vars:,} reference, same order: {same_genes}")
    print(f"junctions: {query['splicing'].n_vars:,} query vs {reference['splicing'].n_vars:,} reference, same order: {same_junctions}")
    if not (same_genes and same_junctions):
        raise ValueError("Bring the query into the reference feature space first: same genes and junctions, same order; "
                         "features the query lacks become zero-count columns (psi_mask = 0).")
    observed = np.asarray((query["splicing"].layers["psi_mask"] > 0).sum(0)).ravel()
    print(f"junctions observed in at least one query unit: {(observed > 0).sum():,} ({(observed > 0).mean():.0%})")


def junction_var_names(mdata, junction_ids):
    # junction ids (chrom_start_end_strand) -> the splicing modality's var_names, which the model's outputs use as columns
    var = mdata["splicing"].var
    lookup = dict(zip(var["junction_id"].astype(str), var.index))
    return [lookup[j] for j in junction_ids]


def raw_psi(mdata, groups, junction_ids):
    # Pooled raw PSI per group: junction reads / reads of its event, summed over the group's units
    cols = [mdata["splicing"].var_names.get_loc(v) for v in junction_var_names(mdata, junction_ids)]
    K = mdata["splicing"].layers["cell_by_junction_matrix"]
    N = mdata["splicing"].layers["cell_by_cluster_matrix"]
    rows = {}
    for group in pd.unique(groups):
        idx = np.where(groups == group)[0]
        k = np.asarray(K[idx][:, cols].sum(0)).ravel()
        n = np.asarray(N[idx][:, cols].sum(0)).ravel()
        units = np.asarray((N[idx][:, cols] > 0).sum(0)).ravel()
        rows[group] = {**{f"psi {j}": (kk / nn if nn > 0 else np.nan) for j, kk, nn in zip(junction_ids, k, n)},
                       **{f"event reads {j}": nn for j, nn in zip(junction_ids, n)},
                       **{f"units with reads {j}": u for j, u in zip(junction_ids, units)}}
    return pd.DataFrame(rows).T


def plot_history(history):
    h = history[history.epoch >= 0]
    fig, ax = plt.subplots(figsize=(4.5, 3))
    ax.axhline(history.validation_loss.iloc[0], color="gray", ls=":", label="reference model (before)")
    ax.plot(h.epoch, h.validation_loss, color="teal", label="fine-tuned decoders")
    ax.set_xlabel("epoch"); ax.set_ylabel("validation loss (query units)"); ax.legend(frameon=False)
    plt.show()


def plot_psi_comparison(model_psi, observed_psi, labels, title):
    # model_psi / observed_psi: DataFrames, rows = groups, columns = junctions (same order as labels)
    fig, axes = plt.subplots(1, len(labels), figsize=(3.2 * len(labels), 3), sharey=True)
    for ax, column, label in zip(np.atleast_1d(axes), model_psi.columns, labels):
        x = np.arange(len(model_psi))
        ax.bar(x - 0.2, model_psi[column], 0.4, color="teal", label="SpliceVI")
        ax.bar(x + 0.2, observed_psi[column], 0.4, color="lightgray", edgecolor="gray", label="raw reads (pooled)")
        ax.set_xticks(x, [g.replace("_", " ") for g in model_psi.index], rotation=20, ha="right")
        ax.set_title(label, fontsize=9); ax.set_ylim(0, 1)
    np.atleast_1d(axes)[0].set_ylabel("PSI"); np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

## 1. Load the data and check the feature space

Each query *unit* here is a metacell; the same steps work for single cells. The splicing modality carries the layers
SpliceVI expects: junction read counts, event read counts, PSI and an observed-PSI mask (see *Data format* in the
docs).

In [ ]:
query = load_mudata(QUERY_FILE)
reference = load_mudata(REFERENCE_FILE)
print(f"query: {query.n_obs:,} units; reference replay: {reference.n_obs:,} cells")
print(query.obs["broad_cell_type"].astype(str).value_counts().to_string())
check_feature_space(query, reference)

EasySci is sparse: most junctions have no reads in most units. The model's job is to share information across junctions
and across similar units.

In [ ]:
observed_per_unit = np.asarray((query["splicing"].layers["psi_mask"] > 0).sum(1)).ravel()
observed_per_cell = np.asarray((reference["splicing"].layers["psi_mask"] > 0).sum(1)).ravel()
print(f"junctions with an observed PSI, median per unit: EasySci metacell {np.median(observed_per_unit):,.0f}, "
      f"Allen cell {np.median(observed_per_cell):,.0f} (of {query['splicing'].n_vars:,})")

## 2. Fine-tune the decoders

Load the reference model *on the query data*. This registers the query with the model's own setup, the same way
`SPLICEVI.load` always does. Then call `finetune_decoders` with the query and the replay set.

What happens during fine-tuning:
- every step uses 32 query units plus 128 replay cells, with Adam at learning rate 1e-4;
- 10% of the query units are held out, and the best state on them is kept (early stopping);
- the depth inputs are fixed at the reference mean, since depth differs between technologies;
- trained: both decoders and the new technology input. Kept: the encoders, the latent space, the per-event
  concentrations and the gene dispersions.

The first line of the log, epoch −1, is the validation loss of the untouched reference model.

In [ ]:
model = SPLICEVI.load(BASE_MODEL, adata=query)
model.to_device(device)
history = model.finetune_decoders(query, reference, max_epochs=MAX_EPOCHS)
plot_history(history)
model.finetune_params_

## 3. Save and reload

The fine-tuned model is a regular SpliceVI model: the technology input and the fine-tuning record are saved with it.
After loading, it decodes as query data (technology input = 1).

In [ ]:
model.save(OUT_DIR, overwrite=True)
model = SPLICEVI.load(OUT_DIR, adata=query)
model.to_device(device)
print("decoder covariate after loading:", model.module.decoder_covariate_value.tolist())

## 4. The query cells in the latent space

The encoders were not changed, so this is the reference model's latent space, the one the Allen cells live in. Cell
types should still separate, even though the model was never trained on EasySci.

In [ ]:
query.obsm["X_splicevi"] = model.get_latent_representation(adata=query, modality="joint")
sc.pp.neighbors(query, use_rep="X_splicevi", n_neighbors=15)
sc.tl.umap(query, random_state=0)
sc.pl.umap(query, color=["broad_cell_type", "Type"], wspace=0.5, frameon=False)

## 5. Differential splicing between two cell types

`preset("paper")` holds the settings used in the paper:
- DS compares posterior samples of the model's PSI;
- a junction counts as changed in a sample when |ΔPSI| > 0.1;
- calls are made at FDR 0.05 (`is_ds_fdr`).

`filter_read_support` then keeps only calls with reads in at least 3 units of each group. That filter is applied after
calling, so the FDR rule is unchanged.

Purkinje neurons have only ~190 metacells, and most junctions have no reads in most of them. This is where borrowing
from the reference model helps.

In [ ]:
cfg = preset("paper")
print("DS settings:", cfg["ds"], "| read-support filter:", cfg["ds_filter"])
cell_type = query.obs["broad_cell_type"].astype(str).values
ds = model.differential_splicing(adata=query, idx1=cell_type == "Cortical_projection_neurons",
                                 idx2=cell_type == "Purkinje_neurons", silent=True, **cfg["ds"])
calls = filter_read_support(ds[ds.is_ds_fdr], **cfg["ds_filter"])
print(f"{ds.is_ds_fdr.sum():,} DS calls at FDR 0.05 out of {len(ds):,} junctions; {len(calls):,} with read support")
calls.assign(dPSI=calls.est_prob1 - calls.est_prob2).sort_values("dPSI", key=abs, ascending=False)[
    ["junction_id", "gene_name", "est_prob1", "est_prob2", "dPSI", "proba_ds", "n_obs_group1", "n_obs_group2"]].head(10)

**An example: Prkg1.** Prkg1 has two alternative first exons, one for each of the protein's isoforms (PKG Iα and
PKG Iβ). Both are spliced to the same downstream exon, so the two junctions compete. SpliceVI says cortical projection
neurons use the Iβ first exon and Purkinje neurons the Iα first exon. Here are the model's estimates next to the raw
reads, pooled over each cell type's units.

In [ ]:
prkg1 = ["chr19_31585733_31764556_-", "chr19_31585733_31663970_-"]   # exon 1 -> 3 (Iα first exon), exon 2 -> 3 (Iβ first exon)
labels = ["Prkg1 exon 1 → 3 (Iα)", "Prkg1 exon 2 → 3 (Iβ)"]
print(ds.set_index("junction_id").loc[prkg1, ["est_prob1", "est_prob2", "proba_ds", "is_ds_fdr", "n_obs_group1", "n_obs_group2"]])

pair = ["Cortical_projection_neurons", "Purkinje_neurons"]
in_pair = np.isin(cell_type, pair)
observed = raw_psi(query[in_pair], cell_type[in_pair], prkg1).loc[pair]
model_psi = pd.DataFrame(ds.set_index("junction_id").loc[prkg1, ["est_prob1", "est_prob2"]].values.T, index=pair,
                         columns=[f"psi {j}" for j in prkg1])
plot_psi_comparison(model_psi, observed[[f"psi {j}" for j in prkg1]], labels, "Prkg1 alternative first exons")
observed

## 6. Comparing conditions within one cell type

The same call works for any two groups of units, for example 21-month vs 3-month-old mice within cortical projection
neurons. Because fine-tuning fixed the depth inputs, groups that differ in sequencing depth are decoded at the same
reference depth.

Use groups of at least 5 units each. With fewer, the posterior differences are dominated by individual units.

In [ ]:
condition = query.obs["Type"].astype(str).values
idx_old = (cell_type == "Cortical_projection_neurons") & (condition == "21mo")
idx_young = (cell_type == "Cortical_projection_neurons") & (condition == "3mo")
print(f"units: 21mo {idx_old.sum():,}, 3mo {idx_young.sum():,}")
assert min(idx_old.sum(), idx_young.sum()) >= 5, "need at least 5 units per group"
ds_age = model.differential_splicing(adata=query, idx1=idx_old, idx2=idx_young, silent=True, **cfg["ds"])
age_calls = filter_read_support(ds_age[ds_age.is_ds_fdr], **cfg["ds_filter"])
print(f"{ds_age.is_ds_fdr.sum():,} DS calls at FDR 0.05; {len(age_calls):,} with read support")
age_calls.assign(dPSI=age_calls.est_prob1 - age_calls.est_prob2).sort_values("dPSI", key=abs, ascending=False)[
    ["junction_id", "gene_name", "est_prob1", "est_prob2", "dPSI", "n_obs_group1", "n_obs_group2"]].head(10)

Differential expression works the same way, with `model.differential_expression(..., **cfg["de"])` and the
`is_de_fdr_0.05` flag.

## 7. Decoding query cells as reference data

The technology input lets you decode the same latent positions as if they came from the reference technology. Use the
`decoder_covariate` context manager: inside the block the model decodes as reference data, and afterwards it goes back to
query mode. The gap between the two shows what the decoders attribute to the technology rather than to the cell.

In [ ]:
prkg1_vars = junction_var_names(query, prkg1)
rows = {}
for ct in pair:
    idx = np.where(cell_type == ct)[0]
    as_query = model.get_normalized_splicing_DM(adata=query, indices=idx, junction_list=prkg1_vars).mean(0)
    with model.decoder_covariate(0.0):   # decode as reference (Allen, Smart-seq) data
        as_reference = model.get_normalized_splicing_DM(adata=query, indices=idx, junction_list=prkg1_vars).mean(0)
    for j, label in zip(prkg1_vars, labels):
        rows[(ct, label)] = {"decoded as EasySci": as_query[j], "decoded as Allen": as_reference[j]}
pd.DataFrame(rows).T.round(3)

## Next steps

- **Your own atlas.** Bring it into the reference model's feature space (same genes and junctions, same order). Features
  your data lacks become zero-count columns, with `psi_mask` = 0.
- **Full training.** Drop `max_epochs` to use the defaults (up to 300 epochs, early stopping after 30 epochs without
  improvement). The paper's EasySci model stopped at epoch 171, after about 6.5 h on one GPU, with all 28,714 training
  metacells and a 20,000-cell replay set.
- **What not to change.** Re-learning the per-event concentrations and gene dispersions on the query data was tested and
  predicted held-out query reads worse, so `finetune_decoders` keeps them from the reference model.
- **Validation.** In the paper, DS calls were checked against the reads of nuclei the model never saw. If your data
  allows it, hold out part of the nuclei before building units.